#### **Importation des bibliothèques**

In [ ]:
# Utilitaires de base
import builtins
import pandas as pd

# Suivi des expriences (MLflow & DagsHub)
import dagshub
import mlflow

# Scikit-Learn : Sparation des donnes et mtriques d'valuation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    f1_score, 
    fbeta_score, 
    precision_score, 
    recall_score
)

# TensorFlow / Keras : Cration et entranement du modle Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import TextVectorization

#### **DagsHub & MLflow Init**

In [ ]:

# Initialisation de la connexion DagsHub avec les identifiants de votre dépôt et activation du mode MLflow
# PATCH WINDOWS : Force l'utilisation de l'encodage UTF-8 lors de l'écriture des fichiers.
# Ceci corrige l'erreur "charmap codec can't encode characters" causée par le nouveau format d'affichage (summary) de Keras 3
import builtins
_original_open = builtins.open
def _utf8_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode and 'encoding' not in kwargs:
        kwargs['encoding'] = 'utf-8'
    return _original_open(*args, **kwargs)
builtins.open = _utf8_open

dagshub.init(repo_owner='Oscar-AS', repo_name='disaster-tweets-project', mlflow=True)

# Définition du nom du dossier (expérience) dans MLflow où toutes nos métriques seront classées
mlflow.set_experiment("Disaster_Tweets_Niveau_3_et_4")

# Affichage d'un message console pour confirmer que le tracking est bien connecté
print("MLflow activé avec succès sur DagsHub !")


#### Importation des données


In [ ]:
# Chargement des données
# Lecture du fichier CSV depuis le dossier Base et stockage dans la variable 'df'
df = pd.read_csv("Base/tweets_clean.csv")



#### **Séparation des données**

In [ ]:
# Séparation Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['target'], test_size=0.2, random_state=42, stratify=df['target']
)

# Affiche dans la console le nombre de tweets utilisés pour l'entraînement
print(f"Taille de l'entraînement : {len(X_train)}")
# Affiche dans la console le nombre de tweets gardés pour le test
print(f"Taille du test : {len(X_test)}")

#### **Modèles Transformers via Hugging Face**

In [ ]:
# Import de l'objet Dataset de Hugging Face
from datasets import Dataset
# Import de evaluate pour calculer de façon standardisée les métriques d'évaluation
# Import de NumPy
import numpy as np

# Transformation de notre DataFrame d'entraînement Pandas en un objet "Dataset" ultra-optimisé de Hugging Face
hf_train = Dataset.from_pandas(pd.DataFrame({'text': X_train, 'label': y_train}))
# Transformation de notre DataFrame de test Pandas
hf_test = Dataset.from_pandas(pd.DataFrame({'text': X_test, 'label': y_test}))

# Importation de scikit-learn pour calculer facilement le F2-Score et les métriques par classe
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, fbeta_score

# Fonction exécutée à la fin de chaque Epoch par le Trainer pour calculer le score
def compute_metrics(eval_pred):
    # Séparation des probabilités prédites (logits) et des vraies réponses (labels)
    logits, labels = eval_pred
    # L'argmax récupère la classe ayant reçu la plus forte probabilité (0 ou 1)
    predictions = np.argmax(logits, axis=-1)
    
    # Précision et Rappel par classe
    precision_cls = precision_score(labels, predictions, average=None)
    recall_cls = recall_score(labels, predictions, average=None)
    
    # Calcul des métriques globales
    f1 = f1_score(labels, predictions, average="macro")
    f2 = fbeta_score(labels, predictions, beta=2, average="macro")
    accuracy = accuracy_score(labels, predictions)
    
    # Retourner toutes les métriques pour le suivi MLflow (le Trainer ajoutera automatiquement le préfixe "eval_")
    return {
        "f1_macro": f1,
        "f2_score": f2,
        "precision_class_0": precision_cls[0],
        "precision_class_1": precision_cls[1],
        "recall_class_0": recall_cls[0],
        "recall_class_1": recall_cls[1],
        "accuracy": accuracy
    }

# Importation du système d'exploitation
import os
# Paramétrage de la variable d'environnement qui indique à Hugging Face dans quel dossier MLflow il doit écrire
os.environ["MLFLOW_EXPERIMENT_NAME"] = "Disaster_Tweets_Niveau_3_et_4"


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

def train_hf_model(model_id, run_name, batch_size=16, epochs=3):
    print(f"========== Début : {model_id} ==========")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    def tokenize_function(examples):
        # ✅ OPTIMISATION 1 : max_length 128 → 64 (tweets courts, -50% temps)
        return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=64)
    
    tokenized_train = hf_train.map(tokenize_function, batched=True)
    tokenized_test = hf_test.map(tokenize_function, batched=True)
    
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
    
    training_args = TrainingArguments(
        output_dir=f"./results_{run_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        report_to="mlflow",
        run_name=run_name,
        
        # ✅ OPTIMISATION 2 : fp16 — divise la VRAM par 2, +40% vitesse
        fp16=True,
        
        # ✅ OPTIMISATION 3 : sélection sur F2 et non sur la loss
        metric_for_best_model="eval_f2_score",
        greater_is_better=True,
        
        # ✅ OPTIMISATION 4 : warmup pour protéger les poids pré-entraînés
        warmup_ratio=0.1,
        
        # ✅ OPTIMISATION 5 : chargement data en parallèle
        dataloader_num_workers=4,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        compute_metrics=compute_metrics,
        # ✅ OPTIMISATION 6 : arrêt automatique si pas d'amélioration
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    
    trainer.train()

    try:
        # Tente d'enregistrer le modèle HuggingFace dans MLflow pour la mise en production
        components = {"model": trainer.model, "tokenizer": tokenizer}
        mlflow.transformers.log_model(transformers_model=components, artifact_path="model")
    except Exception as e:
        print("Avertissement: L'enregistrement du modèle Transformers dans MLflow a échoué:", e)
    
    # Force MLflow à fermer proprement la session de suivi de ce run
    mlflow.end_run()
    # Affiche la fin dans la console
    print(f"========== Fin de l'entraînement pour {model_id} ==========\n")


#### **Modèle RoBERTa (Robustly Optimized BERT)**

##### **Description du modèle**
C'est la version optimisée de BERT, créée par l'équipe d'Intelligence Artificielle de Facebook/Meta.

##### **Explication du fonctionnement**
RoBERTa possède exactement la même architecture que BERT, mais Meta a découvert que Google avait "sous-entraîné" BERT. RoBERTa a été entraîné beaucoup plus longtemps, sur un volume de texte bien plus massif (incluant des flux de réseaux sociaux et Reddit), et en supprimant certaines tâches d'apprentissage jugées inutiles (comme la prédiction de la phrase suivante).



In [ ]:
# Appel de la fonction pour entraîner RoBERTa
train_hf_model(model_id="roberta-base", run_name="4.3_RoBERTa", epochs=2)
